In [ ]:
from datetime import datetime
import os
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm import tqdm

data_path = '${DEMENTIA_DATA_ROOT}'

In [ ]:
criteria_df = pd.read_csv(os.path.join(data_path, 'medical_data/study_criteria_table.csv'))
criteria_df = criteria_df[['FolderName', 'PatientID', 'EMPI', 'MRN', 'MRN_key', 'LastName',
       'FirstName', 'Sex', 'Age', 'DateOfBirth', 'DateOfVisit', 'TypeOfTest',
       'Path']]
print(criteria_df.shape)

In [ ]:
ID_df = criteria_df[['FolderName','PatientID','EMPI','DateOfVisit']].drop_duplicates().reset_index(drop=True)
print(ID_df.shape)
ID_df

In [ ]:
# Exclusion_Enc
# Exclusion_dT - closest to sleep study time 
# Exclusion_ICD
# CDR_Score
# CDR_dT
# CDR_Note
# Low_CDR_Score
# Low_CDR_dT
# Low_CDR_Note
# MMSE_Score
# MMSE_dT
# MMSE_Note
# High_MMSE_Score
# High_MMSE_dT
# High_MMSE_Note
# MoCA_Score
# MoCA_dT
# MoCA_Note
# High_MoCA_Score
# High_MoCA_dT
# High_MoCA_Note
# Neuropsych_dT 
# Neuropsych_Note
# Dementia_Enc: Number of Encounter Diagnosis for Dementia 
# Dementia_ICD
# Dementia_dT: Days between sleep study and most recent dementia diagnosis 
# Dementia_Med: Medication prescribed related to dementia 
# Dementia_Prob
# MCI_Enc:
# MCI_ICD
# MCI_dT:
# MCI_Med: Medication prescribed related to MCI
# MCI_Prob
# Symptomatic_Enc
# Symptomatic_ICD
# Symptomatic_dT
# AlzD_Enc
# AlzD_ICD
# AlzD_dT - most recent dT
# AlzD_Prob
# VaD_Enc
# VaD_ICD
# VaD_dT
# VaD_Prob
# FTD_Enc
# FTD_ICD
# FTD_dT
# FTD_Prob
# DLB_Enc
# DLB_ICD
# DLB_dT
# DLB_Prob
# PDD_Enc
# PDD_dT
# PDD_ICD
# PDD_Prob

# Import Data

In [ ]:
EDW_Enc_df  = pd.read_csv(os.path.join(data_path, 'medical_data/EDW_EncounterDiagnosis.csv'), encoding='utf-8')
EDW_Prob_df = pd.read_csv(os.path.join(data_path, 'medical_data/EDW_ProblemList.csv'), encoding='utf-8')
EDW_Med_df  = pd.read_csv(os.path.join(data_path, 'medical_data/EDW_MedicationDiagnosis.csv'), encoding='utf-8')
print(EDW_Enc_df.shape)
print(EDW_Prob_df.shape)
print(EDW_Med_df.shape)

In [ ]:
EDW_Prob_df = EDW_Prob_df[~EDW_Prob_df['DiagnosisNM'].astype(str).str.lower().str.contains('family history')].reset_index(drop=True)
print(EDW_Prob_df.shape)

In [ ]:
EDW_Enc_df = EDW_Enc_df.dropna(subset=['DiagnosisNM']).reset_index(drop=True)
print(EDW_Enc_df.shape)

In [ ]:
EDW_Enc_df = EDW_Enc_df[~EDW_Enc_df['DiagnosisNM'].astype(str).str.lower().str.contains('family history')].reset_index(drop=True)
print(EDW_Enc_df.shape)

In [ ]:
RPDR_Enc_df = pd.read_csv(os.path.join(data_path, 'medical_data/RPDR_Enc_All.csv'), encoding='utf-8')
RPDR_Med_df = pd.read_csv(os.path.join(data_path, 'medical_data/RPDR_Med_All.csv'), encoding='utf-8')
RPDR_Dia_df = pd.read_csv(os.path.join(data_path, 'medical_data/RPDR_Dia_All.csv'), encoding='utf-8')
print(RPDR_Enc_df.shape)
print(RPDR_Med_df.shape)
print(RPDR_Dia_df.shape)

In [ ]:
RPDR_Med_df = RPDR_Med_df.dropna(subset=['Medication']).reset_index(drop=True)
EDW_Med_df = EDW_Med_df.dropna(subset=['MedicationDSC']).reset_index(drop=True)
print(RPDR_Med_df.shape)
print(EDW_Med_df.shape)

In [ ]:
RPDR_Med_df = RPDR_Med_df.merge(RPDR_Dia_df[['Diagnosis_Name','Encounter_number']],on=['Encounter_number']).reset_index(drop=True)
print(RPDR_Med_df.shape)

In [ ]:
RPDR_Dia_df = RPDR_Dia_df.dropna(subset=['Diagnosis_Name']).reset_index(drop=True)

In [ ]:
RPDR_Dia_df = RPDR_Dia_df[~RPDR_Dia_df['Diagnosis_Name'].astype(str).str.lower().str.contains('family history')].reset_index(drop=True)
print(RPDR_Dia_df.shape)

In [ ]:
MoCA_EDW_df = pd.read_excel(os.path.join(data_path, 'medical_data/MoCA_EDW_Final.xlsx'))
MoCA_RPDR_df = pd.read_excel(os.path.join(data_path, 'medical_data/MoCA_RPDR_Final.xlsx'))
MMSE_EDW_df = pd.read_excel(os.path.join(data_path, 'medical_data/MMSE_EDW_Final.xlsx'))
MMSE_RPDR_df = pd.read_excel(os.path.join(data_path, 'medical_data/MMSE_RPDR_Final.xlsx'))
CDR_EDW_df = pd.read_excel(os.path.join(data_path, 'medical_data/CDR_EDW_Final.xlsx'))
CDR_RPDR_df = pd.read_excel(os.path.join(data_path, 'medical_data/CDR_RPDR_Final.xlsx'))

In [ ]:
MoCA_EDW_df['ContactDTSForNote'] = [ row['MoCADate'] if not pd.isna(row['MoCADate']) else row['ContactDTSForNote'] for index,row in MoCA_EDW_df.iterrows()]

In [ ]:
MoCA_RPDR_df['LMRNote_Date'] = [ row['MoCADate'] if not pd.isna(row['MoCADate']) else row['LMRNote_Date'] for index,row in MoCA_RPDR_df.iterrows()]

In [ ]:
MMSE_EDW_df['ContactDTSForNote'] = [ row['MMSEDate'] if not pd.isna(row['MMSEDate']) else row['ContactDTSForNote'] for index,row in MMSE_EDW_df.iterrows()]

In [ ]:
Neuropsych_RPDR_df = pd.read_excel(os.path.join(data_path, 'medical_data/Neuropsychiatric Scores/Neuropsych_RPDR_Review.xlsx'))
Neuropsych_EDW_df = pd.read_excel(os.path.join(data_path, 'medical_data/Neuropsychiatric Scores/Neuropsych_EDW_Review.xlsx'))

# Exclusion Criteria

In [ ]:
# Read list of exclusion 
with open(os.path.join(data_path, 'medical_data/exclusion_regex'), "r") as file:
    lines = file.read().split('\n')
exclusion_regex = '|'.join(lines)
exclusion_regex

In [ ]:
RPDR_Exclude_df = RPDR_Dia_df[RPDR_Dia_df['Diagnosis_Name'].astype(str).str.lower().str.contains(exclusion_regex)].reset_index(drop=True)
EDW_Exclude_df = EDW_Enc_df[EDW_Enc_df['DiagnosisNM'].astype(str).str.lower().str.contains(exclusion_regex)].reset_index(drop=True)

In [ ]:
RPDR_Exclude_df = RPDR_Exclude_df.merge(criteria_df[['FolderName','EMPI','DateOfVisit']],on=['EMPI']).reset_index(drop=True)
EDW_Exclude_df = EDW_Exclude_df.merge(criteria_df[['FolderName','PatientID','DateOfVisit']],on=['PatientID']).reset_index(drop=True)

In [ ]:
RPDR_Exclude_df['dT'] = [ (datetime.strptime(row['DateOfVisit'],'%Y-%m-%d') -datetime.strptime(row['Date'],'%m/%d/%Y')).days for index,row in RPDR_Exclude_df.iterrows()]
EDW_Exclude_df['dT'] = [ (datetime.strptime(row['DateOfVisit'],'%Y-%m-%d') -datetime.strptime(row['ContactDTSForEncounter'],'%Y-%m-%d')).days for index,row in EDW_Exclude_df.iterrows()]

In [ ]:
# Exclusion_Enc
# Exclusion_ICD

In [ ]:
criteria_df['Exclusion_Enc'] = 0
criteria_df['Exclusion_ICD'] = ""

In [ ]:
for index, row in tqdm(criteria_df.iterrows(),total=criteria_df.shape[0],position=0,leave=True):
    Study_EDW_Exclude_df = EDW_Exclude_df[EDW_Exclude_df['FolderName'] == row['FolderName']]
    Study_RPDR_Exclude_df = RPDR_Exclude_df[RPDR_Exclude_df['FolderName'] == row['FolderName']]
    criteria_df.at[index,'Exclusion_Enc'] = len(Study_RPDR_Exclude_df.drop_duplicates(subset=['Encounter_number'])) + len(Study_EDW_Exclude_df.drop_duplicates(subset=['PatientEncounterID']))
    ICD_list = list(Study_EDW_Exclude_df['DiagnosisNM']) + list(Study_RPDR_Exclude_df['Diagnosis_Name'])
    if ICD_list != []:
        criteria_df.at[index,'Exclusion_ICD'] = ';'.join(list(set(ICD_list)))
    

# Neuropsychiatric Scores

In [ ]:
# CDR_Score
# CDR_Note
# CDR_dT

In [ ]:
criteria_df['CDR_Score'] = np.nan
criteria_df['CDR_Note'] = ""
criteria_df['CDR_dT'] = np.nan

In [ ]:
Temp_CDR_RPDR_df = CDR_RPDR_df[CDR_RPDR_df['Correct'] == 'Y'][['EMPI','LMRNote_Date','Comments','CDRText','CDRScore']].merge(ID_df)
Temp_CDR_RPDR_df 

In [ ]:
Temp_CDR_RPDR_df['LMRNote_Date'] = Temp_CDR_RPDR_df['LMRNote_Date'].astype(str)
Temp_CDR_RPDR_df['LMRNote_Date']  = [row['LMRNote_Date'].replace('00:00:00','').replace(' ','') for index,row in Temp_CDR_RPDR_df.iterrows()]
Temp_CDR_RPDR_df['dT'] = [ (datetime.strptime(row['DateOfVisit'],'%Y-%m-%d') -datetime.strptime(row['LMRNote_Date'],'%Y-%m-%d')).days for index,row in Temp_CDR_RPDR_df.iterrows()]
Temp_CDR_RPDR_df = Temp_CDR_RPDR_df[Temp_CDR_RPDR_df['dT'] >= -365]
Temp_CDR_RPDR_df['abs_dT'] = [abs(row['dT']) for index,row in Temp_CDR_RPDR_df.iterrows()]
Temp_CDR_RPDR_df = Temp_CDR_RPDR_df.sort_values('abs_dT').drop_duplicates('FolderName')

In [ ]:
Temp_CDR_RPDR_df

In [ ]:
Temp_CDR_RPDR_df = Temp_CDR_RPDR_df[['FolderName','LMRNote_Date', 'CDRScore', 'Comments', 'dT','abs_dT']]
Temp_CDR_RPDR_df.columns = ['FolderName','CDR_Date', 'CDR_Score', 'CDR_Note', 'CDR_dT','abs_dT']
Temp_CDR_RPDR_df

In [ ]:
Temp_CDR_EDW_df = CDR_EDW_df[CDR_EDW_df['Correct'] == 'Y'][['PatientID','ContactDTSForNote','NoteTXT','CDRScore']].merge(ID_df)

In [ ]:
Temp_CDR_EDW_df['ContactDTSForNote'] = Temp_CDR_EDW_df['ContactDTSForNote'].astype(str)
Temp_CDR_EDW_df['ContactDTSForNote']  = [row['ContactDTSForNote'].replace('00:00:00','').replace(' ','') for index,row in Temp_CDR_EDW_df.iterrows()]
Temp_CDR_EDW_df['dT'] = [ (datetime.strptime(row['DateOfVisit'],'%Y-%m-%d') -datetime.strptime(row['ContactDTSForNote'],'%Y-%m-%d')).days for index,row in Temp_CDR_EDW_df.iterrows()]
Temp_CDR_EDW_df = Temp_CDR_EDW_df[Temp_CDR_EDW_df['dT'] >= -365]
Temp_CDR_EDW_df['abs_dT'] = [abs(row['dT']) for index,row in Temp_CDR_EDW_df.iterrows()]
Temp_CDR_EDW_df = Temp_CDR_EDW_df.sort_values('abs_dT').drop_duplicates('FolderName')

In [ ]:
Temp_CDR_EDW_df = Temp_CDR_EDW_df[['FolderName','ContactDTSForNote', 'CDRScore', 'NoteTXT', 'dT','abs_dT']]
Temp_CDR_EDW_df.columns = ['FolderName','CDR_Date', 'CDR_Score', 'CDR_Note', 'CDR_dT','abs_dT']

In [ ]:
Temp_CDR_EDW_df

In [ ]:
Temp_CDR_df = pd.concat([Temp_CDR_RPDR_df,Temp_CDR_EDW_df])

In [ ]:
Temp_CDR_df = Temp_CDR_df.sort_values(by = ['abs_dT']).drop_duplicates(subset=['FolderName'])

In [ ]:
Temp_CDR_df

In [ ]:
for index, row in tqdm(criteria_df.iterrows(),total=criteria_df.shape[0],position=0,leave=True):
    Study_CDR_df = Temp_CDR_df[Temp_CDR_df['FolderName'] == row['FolderName']]
    if len(Study_CDR_df ) != 0:
        criteria_df.at[index,'CDR_Note'] =Study_CDR_df.iloc[0]['CDR_Note']
        criteria_df.at[index,'CDR_dT'] = Study_CDR_df.iloc[0]['CDR_dT']
        criteria_df.at[index,'CDR_Score'] = Study_CDR_df.iloc[0]['CDR_Score']
        

In [ ]:
criteria_df[criteria_df['CDR_Score'] == 0]

In [ ]:
# Low_CDR_Score
# Low_CDR_dT
# Low_CDR_Note

In [ ]:
criteria_df['Low_CDR_Score'] = np.nan
criteria_df['Low_CDR_Note'] = ""
criteria_df['Low_CDR_dT'] = np.nan

In [ ]:
Temp_CDR_RPDR_df = CDR_RPDR_df[CDR_RPDR_df['Correct'] == 'Y'][['EMPI','LMRNote_Date','Comments','CDRText','CDRScore']].merge(ID_df)
Temp_CDR_RPDR_df['LMRNote_Date'] = Temp_CDR_RPDR_df['LMRNote_Date'].astype(str)
Temp_CDR_RPDR_df['LMRNote_Date']  = [row['LMRNote_Date'].replace('00:00:00','').replace(' ','') for index,row in Temp_CDR_RPDR_df.iterrows()]
Temp_CDR_RPDR_df['dT'] = [ (datetime.strptime(row['DateOfVisit'],'%Y-%m-%d') -datetime.strptime(row['LMRNote_Date'],'%Y-%m-%d')).days for index,row in Temp_CDR_RPDR_df.iterrows()]
#Temp_CDR_RPDR_df = Temp_CDR_RPDR_df[Temp_CDR_RPDR_df['dT'] < 0]
#Temp_CDR_RPDR_df = Temp_CDR_RPDR_df.sort_values('CDRScore').drop_duplicates('FolderName')
Temp_CDR_RPDR_df = Temp_CDR_RPDR_df[['FolderName','LMRNote_Date', 'CDRScore', 'Comments', 'dT']]
Temp_CDR_RPDR_df.columns = ['FolderName','CDR_Date', 'CDR_Score', 'CDR_Note', 'CDR_dT']
#Temp_CDR_RPDR_df

In [ ]:
Temp_CDR_EDW_df = CDR_EDW_df[CDR_EDW_df['Correct'] == 'Y'][['PatientID','ContactDTSForNote','NoteTXT','CDRScore']].merge(ID_df)
Temp_CDR_EDW_df['ContactDTSForNote'] = Temp_CDR_EDW_df['ContactDTSForNote'].astype(str)
Temp_CDR_EDW_df['ContactDTSForNote']  = [row['ContactDTSForNote'].replace('00:00:00','').replace(' ','') for index,row in Temp_CDR_EDW_df.iterrows()]
Temp_CDR_EDW_df['dT'] = [ (datetime.strptime(row['DateOfVisit'],'%Y-%m-%d') -datetime.strptime(row['ContactDTSForNote'],'%Y-%m-%d')).days for index,row in Temp_CDR_EDW_df.iterrows()]
#Temp_CDR_EDW_df = Temp_CDR_EDW_df[Temp_CDR_EDW_df['dT'] <0 -365]
#Temp_CDR_EDW_df = Temp_CDR_EDW_df.sort_values('CDRScore').drop_duplicates('FolderName')
Temp_CDR_EDW_df = Temp_CDR_EDW_df[['FolderName','ContactDTSForNote', 'CDRScore', 'NoteTXT', 'dT']]
Temp_CDR_EDW_df.columns = ['FolderName','CDR_Date', 'CDR_Score', 'CDR_Note', 'CDR_dT']

In [ ]:
Temp_CDR_df = pd.concat([Temp_CDR_RPDR_df,Temp_CDR_EDW_df])
#Temp_CDR_df = Temp_CDR_df.sort_values(by = ['CDR_Score']).drop_duplicates(subset=['FolderName'])

In [ ]:
criteria_df

In [ ]:
for index, row in tqdm(criteria_df.iterrows(),total=criteria_df.shape[0],position=0,leave=True):
    Study_CDR_df = Temp_CDR_df[Temp_CDR_df['FolderName'] == row['FolderName']]
    Study_CDR_df = Study_CDR_df[ (Study_CDR_df['CDR_dT'] <= row['CDR_dT'])] #CDRs after closest CDR
    if len(Study_CDR_df ) != 0:
        Study_CDR_df = Study_CDR_df.sort_values(by = ['CDR_Score']).drop_duplicates(subset=['FolderName'])
        criteria_df.at[index,'Low_CDR_Note'] =Study_CDR_df.iloc[0]['CDR_Note']
        criteria_df.at[index,'Low_CDR_dT'] = Study_CDR_df.iloc[0]['CDR_dT']
        criteria_df.at[index,'Low_CDR_Score'] = Study_CDR_df.iloc[0]['CDR_Score']

In [ ]:
# MMSE_Score
# MMSE_Note
# MMSE_dT

In [ ]:
criteria_df['MMSE_Score'] = np.nan
criteria_df['MMSE_Note'] = ""
criteria_df['MMSE_dT'] = np.nan

In [ ]:
MMSE_RPDR_df

In [ ]:
Temp_MMSE_RPDR_df = MMSE_RPDR_df[MMSE_RPDR_df['Correct'] == 'Y'][['EMPI','LMRNote_Date','Comments','MMSEText','MMSEScore']].merge(ID_df)
Temp_MMSE_RPDR_df 

In [ ]:
Temp_MMSE_RPDR_df

In [ ]:
Temp_MMSE_RPDR_df['DateOfVisit'] = Temp_MMSE_RPDR_df['DateOfVisit'].astype(str)
Temp_MMSE_RPDR_df['LMRNote_Date'] = Temp_MMSE_RPDR_df['LMRNote_Date'].astype(str)
Temp_MMSE_RPDR_df['dT'] = [ (datetime.strptime(row['DateOfVisit'],'%Y-%m-%d') -datetime.strptime(row['LMRNote_Date'],'%Y-%m-%d')).days for index,row in Temp_MMSE_RPDR_df.iterrows()]
Temp_MMSE_RPDR_df = Temp_MMSE_RPDR_df[Temp_MMSE_RPDR_df['dT'] >= -365]
Temp_MMSE_RPDR_df['abs_dT'] = [abs(row['dT']) for index,row in Temp_MMSE_RPDR_df.iterrows()]
Temp_MMSE_RPDR_df = Temp_MMSE_RPDR_df.sort_values('abs_dT').drop_duplicates(subset='FolderName')

In [ ]:
Temp_MMSE_RPDR_df

In [ ]:
Temp_MMSE_RPDR_df = Temp_MMSE_RPDR_df[['FolderName','LMRNote_Date', 'MMSEScore', 'Comments', 'dT','abs_dT']]
Temp_MMSE_RPDR_df.columns = ['FolderName','MMSE_Date', 'MMSE_Score', 'MMSE_Note', 'MMSE_dT','abs_dT']
Temp_MMSE_RPDR_df

In [ ]:
MMSE_EDW_df

In [ ]:
Temp_MMSE_EDW_df = MMSE_EDW_df[MMSE_EDW_df['Correct'] == 'Y'][['PatientID','ContactDTSForNote','NoteTXT','MMSEScore']].merge(ID_df)

In [ ]:
Temp_MMSE_EDW_df['ContactDTSForNote'] = Temp_MMSE_EDW_df['ContactDTSForNote'].astype(str)
Temp_MMSE_EDW_df['ContactDTSForNote']  = [row['ContactDTSForNote'].replace('00:00:00','').replace(' ','') for index,row in Temp_MMSE_EDW_df.iterrows()]
Temp_MMSE_EDW_df['dT'] = [ (datetime.strptime(row['DateOfVisit'],'%Y-%m-%d') -datetime.strptime(row['ContactDTSForNote'],'%Y-%m-%d')).days for index,row in Temp_MMSE_EDW_df.iterrows()]
Temp_MMSE_EDW_df = Temp_MMSE_EDW_df[Temp_MMSE_EDW_df['dT'] >= -365]
Temp_MMSE_EDW_df['abs_dT'] = [abs(row['dT']) for index,row in Temp_MMSE_EDW_df.iterrows()]
Temp_MMSE_EDW_df = Temp_MMSE_EDW_df.sort_values('abs_dT').drop_duplicates(subset=['FolderName'])


In [ ]:
Temp_MMSE_EDW_df

In [ ]:
Temp_MMSE_EDW_df = Temp_MMSE_EDW_df[['FolderName','ContactDTSForNote', 'MMSEScore', 'NoteTXT', 'dT','abs_dT']]
Temp_MMSE_EDW_df.columns = ['FolderName','MMSE_Date', 'MMSE_Score', 'MMSE_Note', 'MMSE_dT','abs_dT']

In [ ]:
Temp_MMSE_EDW_df

In [ ]:
Temp_MMSE_df = pd.concat([Temp_MMSE_RPDR_df,Temp_MMSE_EDW_df])

In [ ]:
Temp_MMSE_df = Temp_MMSE_df.sort_values(by = ['abs_dT']).drop_duplicates(subset=['FolderName'])
Temp_MMSE_df

In [ ]:
for index, row in tqdm(criteria_df.iterrows(),total=criteria_df.shape[0],position=0,leave=True):
    Study_MMSE_df = Temp_MMSE_df[Temp_MMSE_df['FolderName'] == row['FolderName']]
    if len(Study_MMSE_df ) != 0:
        criteria_df.at[index,'MMSE_Note'] =Study_MMSE_df.iloc[0]['MMSE_Note']
        criteria_df.at[index,'MMSE_dT'] = Study_MMSE_df.iloc[0]['MMSE_dT']
        criteria_df.at[index,'MMSE_Score'] = Study_MMSE_df.iloc[0]['MMSE_Score']

In [ ]:
criteria_df[criteria_df['MMSE_Score'] == 30]

In [ ]:
#High_MMSE_Score
#High_MMSE_Note
#High_Mmse_dT

In [ ]:
criteria_df['High_MMSE_Score'] = np.nan
criteria_df['High_MMSE_Note'] = ""
criteria_df['High_MMSE_dT'] = np.nan

In [ ]:
Temp_MMSE_RPDR_df = MMSE_RPDR_df[MMSE_RPDR_df['Correct'] == 'Y'][['EMPI','LMRNote_Date','Comments','MMSEText','MMSEScore']].merge(ID_df)
Temp_MMSE_RPDR_df['DateOfVisit'] = Temp_MMSE_RPDR_df['DateOfVisit'].astype(str)
Temp_MMSE_RPDR_df['LMRNote_Date'] = Temp_MMSE_RPDR_df['LMRNote_Date'].astype(str)
Temp_MMSE_RPDR_df['dT'] = [ (datetime.strptime(row['DateOfVisit'],'%Y-%m-%d') -datetime.strptime(row['LMRNote_Date'],'%Y-%m-%d')).days for index,row in Temp_MMSE_RPDR_df.iterrows()]
#Temp_MMSE_RPDR_df = Temp_MMSE_RPDR_df[Temp_MMSE_RPDR_df['dT'] < 0]
#Temp_MMSE_RPDR_df = Temp_MMSE_RPDR_df.sort_values('MMSEScore',ascending=False).drop_duplicates(subset='FolderName')
Temp_MMSE_RPDR_df = Temp_MMSE_RPDR_df[['FolderName','LMRNote_Date', 'MMSEScore', 'Comments', 'dT']]
Temp_MMSE_RPDR_df.columns = ['FolderName','MMSE_Date', 'MMSE_Score', 'MMSE_Note', 'MMSE_dT']

In [ ]:
Temp_MMSE_EDW_df = MMSE_EDW_df[MMSE_EDW_df['Correct'] == 'Y'][['PatientID','ContactDTSForNote','NoteTXT','MMSEScore']].merge(ID_df)
Temp_MMSE_EDW_df['ContactDTSForNote'] = Temp_MMSE_EDW_df['ContactDTSForNote'].astype(str)
Temp_MMSE_EDW_df['ContactDTSForNote']  = [row['ContactDTSForNote'].replace('00:00:00','').replace(' ','') for index,row in Temp_MMSE_EDW_df.iterrows()]
Temp_MMSE_EDW_df['dT'] = [ (datetime.strptime(row['DateOfVisit'],'%Y-%m-%d') -datetime.strptime(row['ContactDTSForNote'],'%Y-%m-%d')).days for index,row in Temp_MMSE_EDW_df.iterrows()]
#Temp_MMSE_EDW_df = Temp_MMSE_EDW_df[Temp_MMSE_EDW_df['dT'] < 0]
#Temp_MMSE_EDW_df = Temp_MMSE_EDW_df.sort_values('MMSEScore',ascending=False).drop_duplicates(subset=['FolderName'])
Temp_MMSE_EDW_df = Temp_MMSE_EDW_df[['FolderName','ContactDTSForNote', 'MMSEScore', 'NoteTXT', 'dT']]
Temp_MMSE_EDW_df.columns = ['FolderName','MMSE_Date', 'MMSE_Score', 'MMSE_Note', 'MMSE_dT']

In [ ]:
Temp_MMSE_df = pd.concat([Temp_MMSE_RPDR_df,Temp_MMSE_EDW_df])
#Temp_MMSE_df = Temp_MMSE_df.sort_values(by = ['MMSEScore'],ascending=False).drop_duplicates(subset=['FolderName'])

In [ ]:
for index, row in tqdm(criteria_df[criteria_df['FolderName'].str.contains('Oneill_Olivia_101917_2306.000')].iterrows(),total=criteria_df.shape[0],position=0,leave=True):
    Study_MMSE_df = Temp_MMSE_df[Temp_MMSE_df['FolderName'] == row['FolderName']]
    Study_MMSE_df = Study_MMSE_df[Study_MMSE_df['MMSE_dT']<= row['MMSE_dT']]
    if len(Study_MMSE_df ) != 0:
        Study_MMSE_df = Study_MMSE_df.sort_values(by = ['MMSE_Score'],ascending=False).drop_duplicates(subset=['FolderName'])
        criteria_df.at[index,'High_MMSE_Note'] =Study_MMSE_df.iloc[0]['MMSE_Note']
        criteria_df.at[index,'High_MMSE_dT'] = Study_MMSE_df.iloc[0]['MMSE_dT']
        criteria_df.at[index,'High_MMSE_Score'] = Study_MMSE_df.iloc[0]['MMSE_Score']

In [ ]:
# MoCA_Score
# MoCA_Note
# MoCA_dT

In [ ]:
criteria_df['MoCA_Score'] = np.nan
criteria_df['MoCA_Note'] = ""
criteria_df['MoCA_dT'] = np.nan

In [ ]:
MoCA_RPDR_df

In [ ]:
Temp_MoCA_RPDR_df = MoCA_RPDR_df[MoCA_RPDR_df['Correct'] == 'Y'][['EMPI','LMRNote_Date','Comments','MoCAText','MoCAScore']].merge(ID_df)
Temp_MoCA_RPDR_df 

In [ ]:
Temp_MoCA_RPDR_df['DateOfVisit'] = Temp_MoCA_RPDR_df['DateOfVisit'].astype(str)
Temp_MoCA_RPDR_df['LMRNote_Date'] = Temp_MoCA_RPDR_df['LMRNote_Date'].astype(str)
Temp_MoCA_RPDR_df['LMRNote_Date']  = [row['LMRNote_Date'].replace('00:00:00','').replace(' ','') for index,row in Temp_MoCA_RPDR_df.iterrows()]
Temp_MoCA_RPDR_df['dT'] = [ (datetime.strptime(row['DateOfVisit'],'%Y-%m-%d') -datetime.strptime(row['LMRNote_Date'],'%Y-%m-%d')).days for index,row in Temp_MoCA_RPDR_df.iterrows()]
Temp_MoCA_RPDR_df = Temp_MoCA_RPDR_df[Temp_MoCA_RPDR_df['dT'] >= -365]
Temp_MoCA_RPDR_df['abs_dT'] = [abs(row['dT']) for index,row in Temp_MoCA_RPDR_df.iterrows()]
Temp_MoCA_RPDR_df = Temp_MoCA_RPDR_df.sort_values('abs_dT').drop_duplicates(subset='FolderName')

In [ ]:
Temp_MoCA_RPDR_df

In [ ]:
Temp_MoCA_RPDR_df = Temp_MoCA_RPDR_df[['FolderName','LMRNote_Date', 'MoCAScore', 'Comments', 'dT','abs_dT']]
Temp_MoCA_RPDR_df.columns = ['FolderName','MoCA_Date', 'MoCA_Score', 'MoCA_Note', 'MoCA_dT','abs_dT']
Temp_MoCA_RPDR_df

In [ ]:
Temp_MoCA_EDW_df = MoCA_EDW_df[MoCA_EDW_df['Correct'] == 'Y'][['PatientID','ContactDTSForNote','NoteTXT','MoCAScore']].merge(ID_df)

In [ ]:
Temp_MoCA_EDW_df['ContactDTSForNote'] = Temp_MoCA_EDW_df['ContactDTSForNote'].astype(str)
Temp_MoCA_EDW_df['ContactDTSForNote']  = [row['ContactDTSForNote'].replace('00:00:00','').replace(' ','') for index,row in Temp_MoCA_EDW_df.iterrows()]
Temp_MoCA_EDW_df['dT'] = [ (datetime.strptime(row['DateOfVisit'],'%Y-%m-%d') -datetime.strptime(row['ContactDTSForNote'],'%Y-%m-%d')).days for index,row in Temp_MoCA_EDW_df.iterrows()]
Temp_MoCA_EDW_df = Temp_MoCA_EDW_df[Temp_MoCA_EDW_df['dT'] >= -365]
Temp_MoCA_EDW_df['abs_dT'] = [abs(row['dT']) for index,row in Temp_MoCA_EDW_df.iterrows()]
Temp_MoCA_EDW_df = Temp_MoCA_EDW_df.sort_values('abs_dT').drop_duplicates(subset=['FolderName'])

In [ ]:
Temp_MoCA_EDW_df

In [ ]:
Temp_MoCA_EDW_df = Temp_MoCA_EDW_df[['FolderName','ContactDTSForNote', 'MoCAScore', 'NoteTXT', 'dT','abs_dT']]
Temp_MoCA_EDW_df.columns = ['FolderName','MoCA_Date', 'MoCA_Score', 'MoCA_Note', 'MoCA_dT','abs_dT']

In [ ]:
Temp_MoCA_EDW_df

In [ ]:
Temp_MoCA_df = pd.concat([Temp_MoCA_RPDR_df,Temp_MoCA_EDW_df])

In [ ]:
Temp_MoCA_df = Temp_MoCA_df.sort_values(by = ['abs_dT']).drop_duplicates(subset=['FolderName'])
Temp_MoCA_df

In [ ]:
for index, row in tqdm(criteria_df.iterrows(),total=criteria_df.shape[0],position=0,leave=True):
    Study_MoCA_df = Temp_MoCA_df[Temp_MoCA_df['FolderName'] == row['FolderName']]
    if len(Study_MoCA_df ) != 0:
        criteria_df.at[index,'MoCA_Note'] =Study_MoCA_df.iloc[0]['MoCA_Note']
        criteria_df.at[index,'MoCA_dT'] = Study_MoCA_df.iloc[0]['MoCA_dT']
        criteria_df.at[index,'MoCA_Score'] = Study_MoCA_df.iloc[0]['MoCA_Score']

In [ ]:
criteria_df[criteria_df['MoCA_Score'] < 25]

In [ ]:
#High_MMSE_Score
#High_MMSE_Note
#High_Mmse_dT

In [ ]:
criteria_df['High_MoCA_Score'] = np.nan
criteria_df['High_MoCA_Note'] = ""
criteria_df['High_MoCA_dT'] = np.nan

In [ ]:
Temp_MoCA_RPDR_df = MoCA_RPDR_df[MoCA_RPDR_df['Correct'] == 'Y'][['EMPI','LMRNote_Date','Comments','MoCAText','MoCAScore']].merge(ID_df)
Temp_MoCA_RPDR_df['DateOfVisit'] = Temp_MoCA_RPDR_df['DateOfVisit'].astype(str)
Temp_MoCA_RPDR_df['LMRNote_Date'] = Temp_MoCA_RPDR_df['LMRNote_Date'].astype(str)
Temp_MoCA_RPDR_df['LMRNote_Date']  = [row['LMRNote_Date'].replace('00:00:00','').replace(' ','') for index,row in Temp_MoCA_RPDR_df.iterrows()]
Temp_MoCA_RPDR_df['dT'] = [ (datetime.strptime(row['DateOfVisit'],'%Y-%m-%d') -datetime.strptime(row['LMRNote_Date'],'%Y-%m-%d')).days for index,row in Temp_MoCA_RPDR_df.iterrows()]
#Temp_MoCA_RPDR_df = Temp_MoCA_RPDR_df[Temp_MoCA_RPDR_df['dT'] >= -365]
#Temp_MoCA_RPDR_df['abs_dT'] = [abs(row['dT']) for index,row in Temp_MoCA_RPDR_df.iterrows()]
#Temp_MoCA_RPDR_df = Temp_MoCA_RPDR_df.sort_values('abs_dT').drop_duplicates(subset='FolderName')
Temp_MoCA_RPDR_df = Temp_MoCA_RPDR_df[['FolderName','LMRNote_Date', 'MoCAScore', 'Comments', 'dT']]
Temp_MoCA_RPDR_df.columns = ['FolderName','MoCA_Date', 'MoCA_Score', 'MoCA_Note', 'MoCA_dT']

In [ ]:
Temp_MoCA_EDW_df = MoCA_EDW_df[MoCA_EDW_df['Correct'] == 'Y'][['PatientID','ContactDTSForNote','NoteTXT','MoCAScore']].merge(ID_df)
Temp_MoCA_EDW_df['ContactDTSForNote'] = Temp_MoCA_EDW_df['ContactDTSForNote'].astype(str)
Temp_MoCA_EDW_df['ContactDTSForNote']  = [row['ContactDTSForNote'].replace('00:00:00','').replace(' ','') for index,row in Temp_MoCA_EDW_df.iterrows()]
Temp_MoCA_EDW_df['dT'] = [ (datetime.strptime(row['DateOfVisit'],'%Y-%m-%d') -datetime.strptime(row['ContactDTSForNote'],'%Y-%m-%d')).days for index,row in Temp_MoCA_EDW_df.iterrows()]
#Temp_MoCA_EDW_df = Temp_MoCA_EDW_df[Temp_MoCA_EDW_df['dT'] >= -365]
#Temp_MoCA_EDW_df['abs_dT'] = [abs(row['dT']) for index,row in Temp_MoCA_EDW_df.iterrows()]
#Temp_MoCA_EDW_df = Temp_MoCA_EDW_df.sort_values('abs_dT').drop_duplicates(subset=['FolderName'])
Temp_MoCA_EDW_df = Temp_MoCA_EDW_df[['FolderName','ContactDTSForNote', 'MoCAScore', 'NoteTXT', 'dT']]
Temp_MoCA_EDW_df.columns = ['FolderName','MoCA_Date', 'MoCA_Score', 'MoCA_Note', 'MoCA_dT']

In [ ]:
Temp_MoCA_df = pd.concat([Temp_MoCA_RPDR_df,Temp_MoCA_EDW_df])

In [ ]:
for index, row in tqdm(criteria_df.iterrows(),total=criteria_df.shape[0],position=0,leave=True):
    Study_MoCA_df = Temp_MoCA_df[Temp_MoCA_df['FolderName'] == row['FolderName']]
    Study_MoCA_df = Study_MoCA_df[Study_MoCA_df['MoCA_dT'] <= row['MoCA_dT']]
    if len(Study_MoCA_df ) != 0:
        Study_MoCA_df = Study_MoCA_df.sort_values(by = ['MoCA_Score'],ascending=False).drop_duplicates(subset=['FolderName'])
        criteria_df.at[index,'MoCA_Note'] =Study_MoCA_df.iloc[0]['MoCA_Note']
        criteria_df.at[index,'MoCA_dT'] = Study_MoCA_df.iloc[0]['MoCA_dT']
        criteria_df.at[index,'MoCA_Score'] = Study_MoCA_df.iloc[0]['MoCA_Score']

In [ ]:
criteria_df[criteria_df['MoCA_Score'] < 25]

In [ ]:
# Neuropsych_Note
# Neuropsych_dT

In [ ]:
Neuropsych_RPDR_df = Neuropsych_RPDR_df.merge(criteria_df[['FolderName','EMPI','DateOfVisit']],on=['EMPI'])
Neuropsych_EDW_df  = Neuropsych_EDW_df.merge(criteria_df[['FolderName','PatientID','DateOfVisit']],on=['PatientID'])

In [ ]:
Neuropsych_RPDR_df['dT'] = [ (datetime.strptime(row['DateOfVisit'],'%Y-%m-%d') -datetime.strptime(row['LMRNote_Date'],'%Y-%m-%d')).days for index,row in Neuropsych_RPDR_df.iterrows()]
Neuropsych_EDW_df['dT'] = [ (datetime.strptime(row['DateOfVisit'],'%Y-%m-%d') -datetime.strptime(row['ContactDTSForNote'],'%Y-%m-%d')).days for index,row in Neuropsych_EDW_df.iterrows()]
Neuropsych_RPDR_df['abs_dT'] = [abs(row['dT']) for index,row in Neuropsych_RPDR_df.iterrows()]
Neuropsych_EDW_df['abs_dT'] = [abs(row['dT']) for index,row in Neuropsych_EDW_df.iterrows()]
Neuropsych_RPDR_df = Neuropsych_RPDR_df.sort_values(by=['abs_dT'])
Neuropsych_EDW_df = Neuropsych_EDW_df.sort_values(by=['abs_dT'])

In [ ]:
Temp_Neuropsych_RPDR_df = Neuropsych_RPDR_df[['FolderName','dT', 'abs_dT','Comments']]
Temp_Neuropsych_RPDR_df.columns = ['FolderName','dT', 'abs_dT','Neuropsych_Note']

In [ ]:
Temp_Neuropsych_EDW_df = Neuropsych_EDW_df[['FolderName','dT', 'abs_dT','NoteTXT']]
Temp_Neuropsych_EDW_df.columns = ['FolderName','dT', 'abs_dT','Neuropsych_Note']

In [ ]:
criteria_df['Neuropsych_Note'] = ""
criteria_df['Neuropsych_dT'] = np.nan

In [ ]:
for index, row in tqdm(criteria_df.iterrows(),total=criteria_df.shape[0],position=0,leave=True):
    Study_Neuropsych_RPDR_df = Temp_Neuropsych_RPDR_df[Temp_Neuropsych_RPDR_df['FolderName'] == row['FolderName']]
    Study_Neuropsych_EDW_df = Temp_Neuropsych_EDW_df[Temp_Neuropsych_EDW_df['FolderName'] == row['FolderName']]
    if (len(Study_Neuropsych_RPDR_df) >= 1)|(len(Study_Neuropsych_EDW_df) >= 1):
        criteria_df.at[index,'Neuropsych_Note'] = pd.concat([Study_Neuropsych_RPDR_df,Study_Neuropsych_EDW_df]).sort_values(by=['abs_dT']).iloc[0]['Neuropsych_Note']
        criteria_df.at[index,'Neuropsych_dT'] = pd.concat([Study_Neuropsych_RPDR_df,Study_Neuropsych_EDW_df]).sort_values(by=['abs_dT']).iloc[0]['dT']

In [ ]:
criteria_df[criteria_df['Neuropsych_Note'] !=""]

# Dementia Criteria

In [ ]:
criteria_df['Dementia_Enc'] = 0
criteria_df['Dementia_dT'] = np.nan
criteria_df['Dementia_ICD'] = ""
criteria_df['Dementia_Med'] = 0
criteria_df['Dementia_Prob'] = 0

# Dementia_Enc: Number of Encounter Diagnosis for Dementia 
# Dementia_ICD
# Dementia_dT: Days between sleep study and most recent dementia diagnosis 
# Dementia_Med: Medication prescribed related to dementia 

In [ ]:
# Read list of exclusion 
file = open(r"..\medical_data\dementia_regex", "r")
lines = file.read().split('\n')
dementia_regex = '|'.join(lines)
dementia_regex

In [ ]:
RPDR_Dementia_df = RPDR_Dia_df[RPDR_Dia_df['Diagnosis_Name'].str.contains(dementia_regex)]
EDW_Dementia_df = EDW_Enc_df[EDW_Enc_df['DiagnosisNM'].str.contains(dementia_regex)]

In [ ]:
RPDR_Dementia_df = RPDR_Dementia_df.merge(criteria_df[['FolderName','EMPI','DateOfVisit']],on=['EMPI'])
EDW_Dementia_df = EDW_Dementia_df.merge(criteria_df[['FolderName','PatientID','DateOfVisit']],on=['PatientID'])

In [ ]:
RPDR_Dementia_df['dT'] = [ (datetime.strptime(row['DateOfVisit'],'%Y-%m-%d') -datetime.strptime(row['Date'],'%m/%d/%Y')).days for index,row in RPDR_Dementia_df.iterrows()]
EDW_Dementia_df['dT'] = [ (datetime.strptime(row['DateOfVisit'],'%Y-%m-%d') -datetime.strptime(row['ContactDTSForEncounter'],'%Y-%m-%d')).days for index,row in EDW_Dementia_df.iterrows()]
RPDR_Dementia_df['abs_dT'] = [abs(row['dT']) for index,row in RPDR_Dementia_df.iterrows()]
EDW_Dementia_df['abs_dT'] = [abs(row['dT']) for index,row in EDW_Dementia_df.iterrows()]
EDW_Dementia_df = EDW_Dementia_df.sort_values(by=['abs_dT'])
RPDR_Dementia_df = RPDR_Dementia_df.sort_values(by=['abs_dT'])

In [ ]:
RPDR_Dementia_df = RPDR_Dementia_df[RPDR_Dementia_df['dT'] >= -365]
EDW_Dementia_df = EDW_Dementia_df[EDW_Dementia_df['dT'] >= -365]

In [ ]:
EDW_Dementia_df

In [ ]:
for index, row in tqdm(criteria_df.iterrows(),total=criteria_df.shape[0],position=0,leave=True):
    Study_EDW_Dementia_df = EDW_Dementia_df[EDW_Dementia_df['FolderName'] == row['FolderName']]
    Study_RPDR_Dementia_df = RPDR_Dementia_df[RPDR_Dementia_df['FolderName'] == row['FolderName']]
    criteria_df.at[index,'Dementia_Enc'] = len(Study_RPDR_Dementia_df.drop_duplicates(subset=['Encounter_number'])) + len(Study_EDW_Dementia_df.drop_duplicates(subset=['PatientEncounterID']))
    ICD_list = list(set(list(Study_EDW_Dementia_df['DiagnosisNM']) + list(Study_RPDR_Dementia_df['Diagnosis_Name'])))
    if ICD_list != []:
        if len(ICD_list) == 1:
            criteria_df.at[index,'Dementia_ICD'] = ICD_list[0]
        else:
            criteria_df.at[index,'Dementia_ICD'] = ';'.join(ICD_list)
        criteria_df.at[index,'Dementia_dT'] = pd.concat([Study_EDW_Dementia_df[['dT','abs_dT']],Study_RPDR_Dementia_df[['dT','abs_dT']]]).sort_values(by=['abs_dT']).iloc[0]['dT']

In [ ]:
criteria_df[criteria_df['Dementia_Enc'] > 1]

In [ ]:
# Read list of exclusion 
file = open(r"..\medical_data\medications_regex.txt", "r")
lines = file.read().split('\n')
med_regex = '|'.join(lines)
med_regex

In [ ]:
RPDR_Medication_df = RPDR_Med_df[RPDR_Med_df['Medication'].str.contains(med_regex)]
EDW_Medication_df = EDW_Med_df[EDW_Med_df['MedicationDSC'].str.contains(med_regex)]

In [ ]:
RPDR_Dementia_Medication_df = RPDR_Medication_df[RPDR_Medication_df['Diagnosis_Name'].str.contains(dementia_regex)]
EDW_Dementia_Medication_df = EDW_Medication_df[EDW_Medication_df['DiagnosisNM'].str.contains(dementia_regex)]

In [ ]:
RPDR_Dementia_Medication_df = RPDR_Dementia_Medication_df.merge(criteria_df[['FolderName','EMPI','DateOfVisit']],on=['EMPI'])
EDW_Dementia_Medication_df = EDW_Dementia_Medication_df.merge(criteria_df[['FolderName','PatientID','DateOfVisit']],on=['PatientID'])

In [ ]:
EDW_Dementia_Medication_df

In [ ]:
RPDR_Dementia_Medication_df['dT'] = [ (datetime.strptime(row['DateOfVisit'],'%Y-%m-%d') -datetime.strptime(row['Medication_Date'],'%Y-%m-%d')).days for index,row in RPDR_Dementia_Medication_df.iterrows()]
EDW_Dementia_Medication_df['dT'] = [ (datetime.strptime(row['DateOfVisit'],'%Y-%m-%d') -datetime.strptime(row['ContactDTSForEncounter'],'%Y-%m-%d')).days for index,row in EDW_Dementia_Medication_df.iterrows()]
RPDR_Dementia_Medication_df = RPDR_Dementia_Medication_df[RPDR_Dementia_Medication_df['dT'] >= -365]
EDW_Dementia_Medication_df = EDW_Dementia_Medication_df[EDW_Dementia_Medication_df['dT'] >= -365]

In [ ]:
RPDR_Dementia_Medication_df

In [ ]:
EDW_Dementia_Medication_df

In [ ]:
for index, row in tqdm(criteria_df.iterrows(),total=criteria_df.shape[0],position=0,leave=True):
    Study_EDW_Dementia_Medication_df = EDW_Dementia_Medication_df[EDW_Dementia_Medication_df['FolderName'] == row['FolderName']]
    Study_RPDR_Dementia_Medication_df = RPDR_Dementia_Medication_df[RPDR_Dementia_Medication_df['FolderName'] == row['FolderName']]
    if (len(Study_RPDR_Dementia_Medication_df) >= 1)|(len(Study_EDW_Dementia_Medication_df) >= 1):
        criteria_df.at[index,'Dementia_Med'] =len(Study_RPDR_Dementia_Medication_df.drop_duplicates(subset=['Encounter_number'])) + len(Study_EDW_Dementia_Medication_df.drop_duplicates(subset=['PatientEncounterID']))
    

In [ ]:
criteria_df[criteria_df['Dementia_Med'] >= 1]

In [ ]:
EDW_Dementia_Prob_df = EDW_Prob_df[EDW_Prob_df['DiagnosisNM'].str.contains(dementia_regex) & (EDW_Prob_df['ProblemStatusDSC'] =='Active') ]

In [ ]:
EDW_Dementia_Prob_df = EDW_Dementia_Prob_df.dropna(subset=['DiagnosisDTS'])

In [ ]:
EDW_Dementia_Prob_df = EDW_Dementia_Prob_df[EDW_Dementia_Prob_df['DiagnosisDTS'].str.contains('-') ]

In [ ]:
EDW_Dementia_Prob_df['DiagnosisDTS'] = EDW_Dementia_Prob_df['DiagnosisDTS'].astype(str)
EDW_Dementia_Prob_df= EDW_Dementia_Prob_df.merge(criteria_df[['FolderName','PatientID','DateOfVisit']],on=['PatientID'])
EDW_Dementia_Prob_df['dT'] = [ (datetime.strptime(row['DateOfVisit'],'%Y-%m-%d') -datetime.strptime(row['DiagnosisDTS'],'%Y-%m-%d')).days for index,row in EDW_Dementia_Prob_df.iterrows()]
EDW_Dementia_Prob_df = EDW_Dementia_Prob_df[EDW_Dementia_Prob_df['dT'] >= -365]

In [ ]:
for index, row in tqdm(criteria_df.iterrows(),total=criteria_df.shape[0],position=0,leave=True):
    Study_EDW_Dementia_Prob_df = EDW_Dementia_Prob_df[EDW_Dementia_Prob_df['FolderName'] == row['FolderName']]
    if len(Study_EDW_Dementia_Prob_df) >= 1:
        criteria_df.at[index,'Dementia_Prob'] =len(Study_EDW_Dementia_Prob_df.drop_duplicates())
    

In [ ]:
criteria_df[criteria_df['Dementia_Prob'] >=1]

# MCI Criteria

In [ ]:
criteria_df['MCI_Enc'] = 0
criteria_df['MCI_dT'] = np.nan
criteria_df['MCI_ICD'] = ""
criteria_df['MCI_Med'] = 0
criteria_df['MCI_Prob'] = 0

In [ ]:
# Read list of exclusion 
file = open(r"..\medical_data\MCI_regex.txt", "r")
lines = file.read().split('\n')
MCI_regex = '|'.join(lines)
MCI_regex

In [ ]:
RPDR_MCI_df = RPDR_Dia_df[RPDR_Dia_df['Diagnosis_Name'].str.contains(MCI_regex)]
EDW_MCI_df = EDW_Enc_df[EDW_Enc_df['DiagnosisNM'].str.contains(MCI_regex)]

In [ ]:
RPDR_MCI_df = RPDR_MCI_df.merge(criteria_df[['FolderName','EMPI','DateOfVisit']],on=['EMPI'])
EDW_MCI_df = EDW_MCI_df.merge(criteria_df[['FolderName','PatientID','DateOfVisit']],on=['PatientID'])

In [ ]:
RPDR_MCI_df

In [ ]:
EDW_MCI_df

In [ ]:
RPDR_MCI_df['dT'] = [ (datetime.strptime(row['DateOfVisit'],'%Y-%m-%d') -datetime.strptime(row['Date'],'%m/%d/%Y')).days for index,row in RPDR_MCI_df.iterrows()]
EDW_MCI_df['dT'] = [ (datetime.strptime(row['DateOfVisit'],'%Y-%m-%d') -datetime.strptime(row['ContactDTSForEncounter'],'%Y-%m-%d')).days for index,row in EDW_MCI_df.iterrows()]
RPDR_MCI_df['abs_dT'] = [abs(row['dT']) for index,row in RPDR_MCI_df.iterrows()]
EDW_MCI_df['abs_dT'] = [abs(row['dT']) for index,row in EDW_MCI_df.iterrows()]
EDW_MCI_df = EDW_MCI_df.sort_values(by=['abs_dT'])
RPDR_MCI_df = RPDR_MCI_df.sort_values(by=['abs_dT'])
RPDR_MCI_df = RPDR_MCI_df[RPDR_MCI_df['dT'] >= -365]
EDW_MCI_df = EDW_MCI_df[EDW_MCI_df['dT'] >= -365]

In [ ]:
for index, row in tqdm(criteria_df.iterrows(),total=criteria_df.shape[0],position=0,leave=True):
    Study_EDW_MCI_df = EDW_MCI_df[EDW_MCI_df['FolderName'] == row['FolderName']]
    Study_RPDR_MCI_df = RPDR_MCI_df[RPDR_MCI_df['FolderName'] == row['FolderName']]
    criteria_df.at[index,'MCI_Enc'] = len(Study_RPDR_MCI_df.drop_duplicates(subset=['Encounter_number'])) + len(Study_EDW_MCI_df.drop_duplicates(subset=['PatientEncounterID']))
    ICD_list = list(set(list(Study_EDW_MCI_df['DiagnosisNM']) + list(Study_RPDR_MCI_df['Diagnosis_Name'])))
    if ICD_list != []:
        if len(ICD_list) == 1:
            criteria_df.at[index,'MCI_ICD'] = ICD_list[0]
        else:
            criteria_df.at[index,'MCI_ICD'] = ';'.join(ICD_list)
        criteria_df.at[index,'MCI_dT'] = pd.concat([Study_EDW_MCI_df[['dT','abs_dT']],Study_RPDR_MCI_df[['dT','abs_dT']]]).sort_values(by=['abs_dT']).iloc[0]['dT']

In [ ]:
criteria_df[criteria_df['MCI_Enc'] >= 1]

In [ ]:
RPDR_MCI_Medication_df = RPDR_Medication_df[RPDR_Medication_df['Diagnosis_Name'].str.contains(MCI_regex)]
EDW_MCI_Medication_df = EDW_Medication_df[EDW_Medication_df['DiagnosisNM'].str.contains(MCI_regex)]

In [ ]:
RPDR_MCI_Medication_df = RPDR_MCI_Medication_df.merge(criteria_df[['FolderName','EMPI','DateOfVisit']],on=['EMPI'])
EDW_MCI_Medication_df = EDW_MCI_Medication_df.merge(criteria_df[['FolderName','PatientID','DateOfVisit']],on=['PatientID'])

In [ ]:
RPDR_MCI_Medication_df['dT'] = [ (datetime.strptime(row['DateOfVisit'],'%Y-%m-%d') -datetime.strptime(row['Medication_Date'],'%Y-%m-%d')).days for index,row in RPDR_MCI_Medication_df.iterrows()]
EDW_MCI_Medication_df['dT'] = [ (datetime.strptime(row['DateOfVisit'],'%Y-%m-%d') -datetime.strptime(row['ContactDTSForEncounter'],'%Y-%m-%d')).days for index,row in EDW_MCI_Medication_df.iterrows()]
RPDR_MCI_Medication_df = RPDR_MCI_Medication_df[RPDR_MCI_Medication_df['dT'] >= -365]
EDW_MCI_Medication_df = EDW_MCI_Medication_df[EDW_MCI_Medication_df['dT'] >= -365]

In [ ]:
EDW_MCI_Medication_df

In [ ]:
RPDR_MCI_Medication_df 

In [ ]:
for index, row in tqdm(criteria_df.iterrows(),total=criteria_df.shape[0],position=0,leave=True):
    Study_EDW_MCI_Medication_df = EDW_MCI_Medication_df[EDW_MCI_Medication_df['FolderName'] == row['FolderName']]
    Study_RPDR_MCI_Medication_df = RPDR_MCI_Medication_df[RPDR_MCI_Medication_df['FolderName'] == row['FolderName']]
    if (len(Study_RPDR_MCI_Medication_df) >= 1)|(len(Study_EDW_MCI_Medication_df) >= 1):
        criteria_df.at[index,'MCI_Med'] =len(Study_RPDR_MCI_Medication_df.drop_duplicates(subset=['Encounter_number'])) + len(Study_EDW_MCI_Medication_df.drop_duplicates(subset=['PatientEncounterID']))
    

In [ ]:
criteria_df[criteria_df['MCI_Med'] >= 1]

In [ ]:
EDW_MCI_Prob_df = EDW_Prob_df[EDW_Prob_df['DiagnosisNM'].str.contains(MCI_regex) & (EDW_Prob_df['ProblemStatusDSC'] =='Active') ]
EDW_MCI_Prob_df = EDW_MCI_Prob_df.dropna(subset=['DiagnosisDTS'])
EDW_MCI_Prob_df = EDW_MCI_Prob_df[EDW_MCI_Prob_df['DiagnosisDTS'].str.contains('-') ]
EDW_MCI_Prob_df['DiagnosisDTS'] = EDW_MCI_Prob_df['DiagnosisDTS'].astype(str)
EDW_MCI_Prob_df= EDW_MCI_Prob_df.merge(criteria_df[['FolderName','PatientID','DateOfVisit']],on=['PatientID'])
EDW_MCI_Prob_df['dT'] = [ (datetime.strptime(row['DateOfVisit'],'%Y-%m-%d') -datetime.strptime(row['DiagnosisDTS'],'%Y-%m-%d')).days for index,row in EDW_MCI_Prob_df.iterrows()]
EDW_MCI_Prob_df = EDW_MCI_Prob_df[EDW_MCI_Prob_df['dT'] >= -365]

In [ ]:
for index, row in tqdm(criteria_df.iterrows(),total=criteria_df.shape[0],position=0,leave=True):
    Study_EDW_MCI_Prob_df = EDW_MCI_Prob_df[EDW_MCI_Prob_df['FolderName'] == row['FolderName']]
    if len(Study_EDW_MCI_Prob_df) >= 1:
        criteria_df.at[index,'MCI_Prob'] =len(Study_EDW_MCI_Prob_df.drop_duplicates())
    

In [ ]:
criteria_df[criteria_df['MCI_Prob'] >=1]

# Symptomatic Criteria

In [ ]:
criteria_df['Symptomatic_Enc'] = 0
criteria_df['Symptomatic_dT'] = np.nan
criteria_df['Symptomatic_ICD'] = ""

In [ ]:
# Symptomatic_Enc
# Symptomatic_ICD
# Symptomatic_dT


In [ ]:
# Read list of exclusion 
file = open(r"..\medical_data\symptomatic_regex.txt", "r")
lines = file.read().split('\n')
symptomatic_regex = '|'.join(lines)
symptomatic_regex

In [ ]:
RPDR_Symptomatic_df = RPDR_Dia_df[RPDR_Dia_df['Diagnosis_Name'].str.contains(symptomatic_regex)]
EDW_Symptomatic_df = EDW_Enc_df[EDW_Enc_df['DiagnosisNM'].str.contains(symptomatic_regex)]

In [ ]:
RPDR_Symptomatic_df = RPDR_Symptomatic_df.merge(criteria_df[['FolderName','EMPI','DateOfVisit']],on=['EMPI'])
EDW_Symptomatic_df = EDW_Symptomatic_df.merge(criteria_df[['FolderName','PatientID','DateOfVisit']],on=['PatientID'])

In [ ]:
RPDR_Symptomatic_df 

In [ ]:
EDW_Symptomatic_df

In [ ]:
RPDR_Symptomatic_df['dT'] = [ (datetime.strptime(row['DateOfVisit'],'%Y-%m-%d') -datetime.strptime(row['Date'],'%m/%d/%Y')).days for index,row in RPDR_Symptomatic_df.iterrows()]
EDW_Symptomatic_df['dT'] = [ (datetime.strptime(row['DateOfVisit'],'%Y-%m-%d') -datetime.strptime(row['ContactDTSForEncounter'],'%Y-%m-%d')).days for index,row in EDW_Symptomatic_df.iterrows()]
RPDR_Symptomatic_df['abs_dT'] = [abs(row['dT']) for index,row in RPDR_Symptomatic_df.iterrows()]
EDW_Symptomatic_df['abs_dT'] = [abs(row['dT']) for index,row in EDW_Symptomatic_df.iterrows()]
EDW_Symptomatic_df = EDW_Symptomatic_df.sort_values(by=['abs_dT'])
RPDR_Symptomatic_df = RPDR_Symptomatic_df.sort_values(by=['abs_dT'])
RPDR_Symptomatic_df = RPDR_Symptomatic_df[RPDR_Symptomatic_df['dT'] >= -365]
EDW_Symptomatic_df = EDW_Symptomatic_df[EDW_Symptomatic_df['dT'] >= -365]

In [ ]:
for index, row in tqdm(criteria_df.iterrows(),total=criteria_df.shape[0],position=0,leave=True):
    Study_EDW_Symptomatic_df = EDW_Symptomatic_df[EDW_Symptomatic_df['FolderName'] == row['FolderName']]
    Study_RPDR_Symptomatic_df = RPDR_Symptomatic_df[RPDR_Symptomatic_df['FolderName'] == row['FolderName']]
    criteria_df.at[index,'Symptomatic_Enc'] = len(Study_RPDR_Symptomatic_df.drop_duplicates(subset=['Encounter_number'])) + len(Study_EDW_Symptomatic_df.drop_duplicates(subset=['PatientEncounterID']))
    ICD_list = list(set(list(Study_EDW_Symptomatic_df['DiagnosisNM']) + list(Study_RPDR_Symptomatic_df['Diagnosis_Name'])))
    if ICD_list != []:
        if len(ICD_list) == 1:
            criteria_df.at[index,'Symptomatic_ICD'] = ICD_list[0]
        else:
            criteria_df.at[index,'Symptomatic_ICD'] = ';'.join(ICD_list)
        criteria_df.at[index,'Symptomatic_dT'] = pd.concat([Study_EDW_Symptomatic_df[['dT','abs_dT']],Study_RPDR_Symptomatic_df[['dT','abs_dT']]]).sort_values(by=['abs_dT']).iloc[0]['dT']

In [ ]:
criteria_df[criteria_df['Symptomatic_Enc'] >= 1 ]

# Dementia Subtypes

In [ ]:
# Note there is no time filter! 

# AlzD_Enc
# AlzD_ICD
# AlzD_dT # Latest time
# AlzD_Prob

In [ ]:
criteria_df['AlzD_Enc'] = 0
criteria_df['AlzD_dT'] = np.nan
criteria_df['AlzD_ICD'] = ""
criteria_df['AlzD_Prob'] = 0
RPDR_AlzD_df = RPDR_Dia_df[RPDR_Dia_df['Diagnosis_Name'].str.contains('[Aa]lzheimer')]
EDW_AlzD_df = EDW_Enc_df[EDW_Enc_df['DiagnosisNM'].str.contains('[Aa]lzheimer')]
RPDR_AlzD_df = RPDR_AlzD_df.merge(criteria_df[['FolderName','EMPI','DateOfVisit']],on=['EMPI'])
EDW_AlzD_df = EDW_AlzD_df.merge(criteria_df[['FolderName','PatientID','DateOfVisit']],on=['PatientID'])
RPDR_AlzD_df['dT'] = [ (datetime.strptime(row['DateOfVisit'],'%Y-%m-%d') -datetime.strptime(row['Date'],'%m/%d/%Y')).days for index,row in RPDR_AlzD_df.iterrows()]
EDW_AlzD_df['dT'] = [ (datetime.strptime(row['DateOfVisit'],'%Y-%m-%d') -datetime.strptime(row['ContactDTSForEncounter'],'%Y-%m-%d')).days for index,row in EDW_AlzD_df.iterrows()]
EDW_AlzD_df = EDW_AlzD_df.sort_values(by=['dT'])
RPDR_AlzD_df = RPDR_AlzD_df.sort_values(by=['dT'])

In [ ]:
for index, row in tqdm(criteria_df.iterrows(),total=criteria_df.shape[0],position=0,leave=True):
    Study_EDW_AlzD_df = EDW_AlzD_df[EDW_AlzD_df['FolderName'] == row['FolderName']]
    Study_RPDR_AlzD_df = RPDR_AlzD_df[RPDR_AlzD_df['FolderName'] == row['FolderName']]
    criteria_df.at[index,'AlzD_Enc'] = len(Study_RPDR_AlzD_df.drop_duplicates(subset=['Encounter_number'])) + len(Study_EDW_AlzD_df.drop_duplicates(subset=['PatientEncounterID']))
    ICD_list = list(set(list(Study_EDW_AlzD_df['DiagnosisNM']) + list(Study_RPDR_AlzD_df['Diagnosis_Name'])))
    if ICD_list != []:
        if len(ICD_list) == 1:
            criteria_df.at[index,'AlzD_ICD'] = ICD_list[0]
        else:
            criteria_df.at[index,'AlzD_ICD'] = ';'.join(ICD_list)
        criteria_df.at[index,'AlzD_dT'] = pd.concat([Study_EDW_AlzD_df[['dT']],Study_RPDR_AlzD_df[['dT']]]).sort_values(by=['dT']).iloc[0]['dT']

In [ ]:
EDW_AlzD_Prob_df = EDW_Prob_df[EDW_Prob_df['DiagnosisNM'].str.contains('[Aa]lzheimer') & (EDW_Prob_df['ProblemStatusDSC'] =='Active') ]
EDW_AlzD_Prob_df = EDW_AlzD_Prob_df.dropna(subset=['DiagnosisDTS'])
EDW_AlzD_Prob_df = EDW_AlzD_Prob_df[EDW_AlzD_Prob_df['DiagnosisDTS'].str.contains('-') ]
EDW_AlzD_Prob_df['DiagnosisDTS'] = EDW_AlzD_Prob_df['DiagnosisDTS'].astype(str)
EDW_AlzD_Prob_df= EDW_AlzD_Prob_df.merge(criteria_df[['FolderName','PatientID','DateOfVisit']],on=['PatientID'])
for index, row in tqdm(criteria_df.iterrows(),total=criteria_df.shape[0],position=0,leave=True):
    Study_EDW_AlzD_Prob_df = EDW_AlzD_Prob_df[EDW_AlzD_Prob_df['FolderName'] == row['FolderName']]
    if len(Study_EDW_AlzD_Prob_df) >= 1:
        criteria_df.at[index,'AlzD_Prob'] =len(Study_EDW_AlzD_Prob_df.drop_duplicates())

In [ ]:
criteria_df[criteria_df['AlzD_Enc'] >=1 ]

In [ ]:
# VaD_Enc
# VaD_ICD
# VaD_dT
# VaD_Prob


In [ ]:
criteria_df['VaD_Enc'] = 0
criteria_df['VaD_dT'] = np.nan
criteria_df['VaD_ICD'] = ""
criteria_df['VaD_Prob'] = 0
RPDR_VaD_df = RPDR_Dia_df[RPDR_Dia_df['Diagnosis_Name'].str.contains('[Vv]ascular.{0,10}[Dd]ementia')]
EDW_VaD_df = EDW_Enc_df[EDW_Enc_df['DiagnosisNM'].str.contains('[Vv]ascular.{0,10}[Dd]ementia')]
RPDR_VaD_df = RPDR_VaD_df.merge(criteria_df[['FolderName','EMPI','DateOfVisit']],on=['EMPI'])
EDW_VaD_df = EDW_VaD_df.merge(criteria_df[['FolderName','PatientID','DateOfVisit']],on=['PatientID'])
RPDR_VaD_df['dT'] = [ (datetime.strptime(row['DateOfVisit'],'%Y-%m-%d') -datetime.strptime(row['Date'],'%m/%d/%Y')).days for index,row in RPDR_VaD_df.iterrows()]
EDW_VaD_df['dT'] = [ (datetime.strptime(row['DateOfVisit'],'%Y-%m-%d') -datetime.strptime(row['ContactDTSForEncounter'],'%Y-%m-%d')).days for index,row in EDW_VaD_df.iterrows()]
EDW_VaD_df = EDW_VaD_df.sort_values(by=['dT'])
RPDR_VaD_df = RPDR_VaD_df.sort_values(by=['dT'])

In [ ]:
EDW_VaD_df

In [ ]:
for index, row in tqdm(criteria_df.iterrows(),total=criteria_df.shape[0],position=0,leave=True):
    Study_EDW_VaD_df = EDW_VaD_df[EDW_VaD_df['FolderName'] == row['FolderName']]
    Study_RPDR_VaD_df = RPDR_VaD_df[RPDR_VaD_df['FolderName'] == row['FolderName']]
    criteria_df.at[index,'VaD_Enc'] = len(Study_RPDR_VaD_df.drop_duplicates(subset=['Encounter_number'])) + len(Study_EDW_VaD_df.drop_duplicates(subset=['PatientEncounterID']))
    ICD_list = list(set(list(Study_EDW_VaD_df['DiagnosisNM']) + list(Study_RPDR_VaD_df['Diagnosis_Name'])))
    if ICD_list != []:
        if len(ICD_list) == 1:
            criteria_df.at[index,'VaD_ICD'] = ICD_list[0]
        else:
            criteria_df.at[index,'VaD_ICD'] = ';'.join(ICD_list)
        criteria_df.at[index,'VaD_dT'] = pd.concat([Study_EDW_VaD_df[['dT']],Study_RPDR_VaD_df[['dT']]]).sort_values(by=['dT']).iloc[0]['dT']

In [ ]:
EDW_VaD_Prob_df = EDW_Prob_df[EDW_Prob_df['DiagnosisNM'].str.contains('[Vv]ascular.{0,10}[Dd]ementia') & (EDW_Prob_df['ProblemStatusDSC'] =='Active') ]
EDW_VaD_Prob_df = EDW_VaD_Prob_df.dropna(subset=['DiagnosisDTS'])
EDW_VaD_Prob_df = EDW_VaD_Prob_df[EDW_VaD_Prob_df['DiagnosisDTS'].str.contains('-') ]
EDW_VaD_Prob_df['DiagnosisDTS'] = EDW_VaD_Prob_df['DiagnosisDTS'].astype(str)
EDW_VaD_Prob_df= EDW_VaD_Prob_df.merge(criteria_df[['FolderName','PatientID','DateOfVisit']],on=['PatientID'])
for index, row in tqdm(criteria_df.iterrows(),total=criteria_df.shape[0],position=0,leave=True):
    Study_EDW_VaD_Prob_df = EDW_VaD_Prob_df[EDW_VaD_Prob_df['FolderName'] == row['FolderName']]
    if len(Study_EDW_VaD_Prob_df) >= 1:
        criteria_df.at[index,'VaD_Prob'] =len(Study_EDW_VaD_Prob_df.drop_duplicates())

In [ ]:
criteria_df[criteria_df['VaD_Enc'] >=1 ]

In [ ]:
# FTD_Enc
# FTD_ICD
# FTD_dT
# FTD_Prob

In [ ]:
criteria_df['FTD_Enc'] = 0
criteria_df['FTD_dT'] = np.nan
criteria_df['FTD_ICD'] = ""
criteria_df['FTD_Prob'] = 0
RPDR_FTD_df = RPDR_Dia_df[RPDR_Dia_df['Diagnosis_Name'].str.contains('[Ff]rontotemporal.{0,10}[Dd]ementia')]
EDW_FTD_df = EDW_Enc_df[EDW_Enc_df['DiagnosisNM'].str.contains('[Ff]rontotemporal.{0,10}[Dd]ementia')]
RPDR_FTD_df = RPDR_FTD_df.merge(criteria_df[['FolderName','EMPI','DateOfVisit']],on=['EMPI'])
EDW_FTD_df = EDW_FTD_df.merge(criteria_df[['FolderName','PatientID','DateOfVisit']],on=['PatientID'])
RPDR_FTD_df['dT'] = [ (datetime.strptime(row['DateOfVisit'],'%Y-%m-%d') -datetime.strptime(row['Date'],'%m/%d/%Y')).days for index,row in RPDR_FTD_df.iterrows()]
EDW_FTD_df['dT'] = [ (datetime.strptime(row['DateOfVisit'],'%Y-%m-%d') -datetime.strptime(row['ContactDTSForEncounter'],'%Y-%m-%d')).days for index,row in EDW_FTD_df.iterrows()]
EDW_FTD_df = EDW_FTD_df.sort_values(by=['dT'])
RPDR_FTD_df = RPDR_FTD_df.sort_values(by=['dT'])

In [ ]:
EDW_FTD_df 

In [ ]:
for index, row in tqdm(criteria_df.iterrows(),total=criteria_df.shape[0],position=0,leave=True):
    Study_EDW_FTD_df = EDW_FTD_df[EDW_FTD_df['FolderName'] == row['FolderName']]
    Study_RPDR_FTD_df = RPDR_FTD_df[RPDR_FTD_df['FolderName'] == row['FolderName']]
    criteria_df.at[index,'FTD_Enc'] = len(Study_RPDR_FTD_df.drop_duplicates(subset=['Encounter_number'])) + len(Study_EDW_FTD_df.drop_duplicates(subset=['PatientEncounterID']))
    ICD_list = list(set(list(Study_EDW_FTD_df['DiagnosisNM']) + list(Study_RPDR_FTD_df['Diagnosis_Name'])))
    if ICD_list != []:
        if len(ICD_list) == 1:
            criteria_df.at[index,'FTD_ICD'] = ICD_list[0]
        else:
            criteria_df.at[index,'FTD_ICD'] = ';'.join(ICD_list)
        criteria_df.at[index,'FTD_dT'] = pd.concat([Study_EDW_FTD_df[['dT']],Study_RPDR_FTD_df[['dT']]]).sort_values(by=['dT']).iloc[0]['dT']

In [ ]:
EDW_FTD_Prob_df = EDW_Prob_df[EDW_Prob_df['DiagnosisNM'].str.contains('[Ff]rontotemporal.{0,10}[Dd]ementia') & (EDW_Prob_df['ProblemStatusDSC'] =='Active') ]
EDW_FTD_Prob_df = EDW_FTD_Prob_df.dropna(subset=['DiagnosisDTS'])
EDW_FTD_Prob_df = EDW_FTD_Prob_df[EDW_FTD_Prob_df['DiagnosisDTS'].str.contains('-') ]
EDW_FTD_Prob_df['DiagnosisDTS'] = EDW_FTD_Prob_df['DiagnosisDTS'].astype(str)
EDW_FTD_Prob_df= EDW_FTD_Prob_df.merge(criteria_df[['FolderName','PatientID','DateOfVisit']],on=['PatientID'])
for index, row in tqdm(criteria_df.iterrows(),total=criteria_df.shape[0],position=0,leave=True):
    Study_EDW_FTD_Prob_df = EDW_FTD_Prob_df[EDW_FTD_Prob_df['FolderName'] == row['FolderName']]
    if len(Study_EDW_FTD_Prob_df) >= 1:
        criteria_df.at[index,'FTD_Prob'] =len(Study_EDW_FTD_Prob_df.drop_duplicates())

In [ ]:
criteria_df[criteria_df['FTD_Enc'] >=1 ]

In [ ]:
# DLB_Enc
# DLB_ICD
# DLB_dT
# DLB_Prob

In [ ]:
criteria_df['DLB_Enc'] = 0
criteria_df['DLB_dT'] = np.nan
criteria_df['DLB_ICD'] = ""
criteria_df['DLB_Prob'] = 0
RPDR_DLB_df = RPDR_Dia_df[RPDR_Dia_df['Diagnosis_Name'].str.contains('[Dd]ementia.{0,20}[Ll]ewy.{0,20}[Bb]od|[Ll]ewy.{0,20}[Bb]od.{0,20}[Dd]ementia')]
EDW_DLB_df = EDW_Enc_df[EDW_Enc_df['DiagnosisNM'].str.contains('[Dd]ementia.{0,20}[Ll]ewy.{0,20}[Bb]od|[Ll]ewy.{0,20}[Bb]od.{0,20}[Dd]ementia')]
RPDR_DLB_df = RPDR_DLB_df.merge(criteria_df[['FolderName','EMPI','DateOfVisit']],on=['EMPI'])
EDW_DLB_df = EDW_DLB_df.merge(criteria_df[['FolderName','PatientID','DateOfVisit']],on=['PatientID'])
RPDR_DLB_df['dT'] = [ (datetime.strptime(row['DateOfVisit'],'%Y-%m-%d') -datetime.strptime(row['Date'],'%m/%d/%Y')).days for index,row in RPDR_DLB_df.iterrows()]
EDW_DLB_df['dT'] = [ (datetime.strptime(row['DateOfVisit'],'%Y-%m-%d') -datetime.strptime(row['ContactDTSForEncounter'],'%Y-%m-%d')).days for index,row in EDW_DLB_df.iterrows()]
EDW_DLB_df = EDW_DLB_df.sort_values(by=['dT'])
RPDR_DLB_df = RPDR_DLB_df.sort_values(by=['dT'])

In [ ]:
EDW_DLB_df 

In [ ]:
for index, row in tqdm(criteria_df.iterrows(),total=criteria_df.shape[0],position=0,leave=True):
    Study_EDW_DLB_df = EDW_DLB_df[EDW_DLB_df['FolderName'] == row['FolderName']]
    Study_RPDR_DLB_df = RPDR_DLB_df[RPDR_DLB_df['FolderName'] == row['FolderName']]
    criteria_df.at[index,'DLB_Enc'] = len(Study_RPDR_DLB_df.drop_duplicates(subset=['Encounter_number'])) + len(Study_EDW_DLB_df.drop_duplicates(subset=['PatientEncounterID']))
    ICD_list = list(set(list(Study_EDW_DLB_df['DiagnosisNM']) + list(Study_RPDR_DLB_df['Diagnosis_Name'])))
    if ICD_list != []:
        if len(ICD_list) == 1:
            criteria_df.at[index,'DLB_ICD'] = ICD_list[0]
        else:
            criteria_df.at[index,'DLB_ICD'] = ';'.join(ICD_list)
        criteria_df.at[index,'DLB_dT'] = pd.concat([Study_EDW_DLB_df[['dT']],Study_RPDR_DLB_df[['dT']]]).sort_values(by=['dT']).iloc[0]['dT']

In [ ]:
EDW_DLB_Prob_df = EDW_Prob_df[EDW_Prob_df['DiagnosisNM'].str.contains('[Dd]ementia.{0,20}[Ll]ewy.{0,20}[Bb]od|[Ll]ewy.{0,20}[Bb]od.{0,20}[Dd]ementia') & (EDW_Prob_df['ProblemStatusDSC'] =='Active') ]
EDW_DLB_Prob_df = EDW_DLB_Prob_df.dropna(subset=['DiagnosisDTS'])
EDW_DLB_Prob_df = EDW_DLB_Prob_df[EDW_DLB_Prob_df['DiagnosisDTS'].str.contains('-') ]
EDW_DLB_Prob_df['DiagnosisDTS'] = EDW_DLB_Prob_df['DiagnosisDTS'].astype(str)
EDW_DLB_Prob_df= EDW_DLB_Prob_df.merge(criteria_df[['FolderName','PatientID','DateOfVisit']],on=['PatientID'])
for index, row in tqdm(criteria_df.iterrows(),total=criteria_df.shape[0],position=0,leave=True):
    Study_EDW_DLB_Prob_df = EDW_DLB_Prob_df[EDW_DLB_Prob_df['FolderName'] == row['FolderName']]
    if len(Study_EDW_DLB_Prob_df) >= 1:
        criteria_df.at[index,'DLB_Prob'] =len(Study_EDW_DLB_Prob_df.drop_duplicates())

In [ ]:
criteria_df[criteria_df['DLB_Enc'] >=1 ]

In [ ]:
# PD_Enc
# PD_dT
# PD_ICD
# PD_Prob

In [ ]:
criteria_df['PD_Enc'] = 0
criteria_df['PD_dT'] = np.nan
criteria_df['PD_ICD'] = ""
criteria_df['PD_Prob'] = 0
RPDR_PD_df = RPDR_Dia_df[RPDR_Dia_df['Diagnosis_Name'].str.contains('[Pp]arkinson.{0,20}[Dd]isease')]
EDW_PD_df = EDW_Enc_df[EDW_Enc_df['DiagnosisNM'].str.contains('[Pp]arkinson.{0,20}[Dd]isease')]
RPDR_PD_df = RPDR_PD_df.merge(criteria_df[['FolderName','EMPI','DateOfVisit']],on=['EMPI'])
EDW_PD_df = EDW_PD_df.merge(criteria_df[['FolderName','PatientID','DateOfVisit']],on=['PatientID'])
RPDR_PD_df['dT'] = [ (datetime.strptime(row['DateOfVisit'],'%Y-%m-%d') -datetime.strptime(row['Date'],'%m/%d/%Y')).days for index,row in RPDR_PD_df.iterrows()]
EDW_PD_df['dT'] = [ (datetime.strptime(row['DateOfVisit'],'%Y-%m-%d') -datetime.strptime(row['ContactDTSForEncounter'],'%Y-%m-%d')).days for index,row in EDW_PD_df.iterrows()]
EDW_PD_df = EDW_PD_df.sort_values(by=['dT'])
RPDR_PD_df = RPDR_PD_df.sort_values(by=['dT'])

In [ ]:
RPDR_PD_df

In [ ]:
for index, row in tqdm(criteria_df.iterrows(),total=criteria_df.shape[0],position=0,leave=True):
    Study_EDW_PD_df = EDW_PD_df[EDW_PD_df['FolderName'] == row['FolderName']]
    Study_RPDR_PD_df = RPDR_PD_df[RPDR_PD_df['FolderName'] == row['FolderName']]
    criteria_df.at[index,'PD_Enc'] = len(Study_RPDR_PD_df.drop_duplicates(subset=['Encounter_number'])) + len(Study_EDW_PD_df.drop_duplicates(subset=['PatientEncounterID']))
    ICD_list = list(set(list(Study_EDW_PD_df['DiagnosisNM']) + list(Study_RPDR_PD_df['Diagnosis_Name'])))
    if ICD_list != []:
        if len(ICD_list) == 1:
            criteria_df.at[index,'PD_ICD'] = ICD_list[0]
        else:
            criteria_df.at[index,'PD_ICD'] = ';'.join(ICD_list)
        criteria_df.at[index,'PD_dT'] = pd.concat([Study_EDW_PD_df[['dT']],Study_RPDR_PD_df[['dT']]]).sort_values(by=['dT']).iloc[0]['dT']

In [ ]:
EDW_PD_Prob_df = EDW_Prob_df[EDW_Prob_df['DiagnosisNM'].str.contains('[Pp]arkinson.{0,20}[Dd]isease') & (EDW_Prob_df['ProblemStatusDSC'] =='Active') ]
EDW_PD_Prob_df = EDW_PD_Prob_df.dropna(subset=['DiagnosisDTS'])
EDW_PD_Prob_df = EDW_PD_Prob_df[EDW_PD_Prob_df['DiagnosisDTS'].str.contains('-') ]
EDW_PD_Prob_df['DiagnosisDTS'] = EDW_PD_Prob_df['DiagnosisDTS'].astype(str)
EDW_PD_Prob_df= EDW_PD_Prob_df.merge(criteria_df[['FolderName','PatientID','DateOfVisit']],on=['PatientID'])
for index, row in tqdm(criteria_df.iterrows(),total=criteria_df.shape[0],position=0,leave=True):
    Study_EDW_PD_Prob_df = EDW_PD_Prob_df[EDW_PD_Prob_df['FolderName'] == row['FolderName']]
    if len(Study_EDW_PD_Prob_df) >= 1:
        criteria_df.at[index,'PD_Prob'] =len(Study_EDW_PD_Prob_df.drop_duplicates())

In [ ]:
criteria_df[criteria_df['PD_Enc'] >=1 ]

In [ ]:
label_columns = ['True_Stage',
'True_Certainty',
'True_Diease',
 'Note'  ,
'Predicted_Stage',
'Predicted_Certainty',
'Predicted_dT',
'Predicted_Disease']
label_array = np.empty((len(criteria_df),8))
label_array[:] = np.NaN
label_df = pd.DataFrame(data=label_array,columns=label_columns)

In [ ]:
criteria_df = pd.concat([label_df,criteria_df],axis=1) 

In [ ]:
criteria_df.to_excel(r'..\medical_data\study_medical_table_V3.xlsx',index=False)

In [ ]:
medical_df = pd.read_excel(r'..\medical_data\study_medical_table_V3.xlsx')

In [ ]:
medical_df[medical_df['FolderName'].str.contains('Oneill_Olivia_101917_2306.000')]['MMSE_Score']